# Project 1: News Classification Across Datasets
Run all cells top to bottom. Data is loaded from the `dataset/` folder of the GitHub repo cloned in the first cell.

In [ ]:
import os
from google.colab import userdata

REPO_URL = "https://github.com/OWNER-USERNAME/REPO-NAME.git"  # <-- copy from the repo page
TOKEN = userdata.get("GITHUB_TOKEN")  # stored in Colab secrets, not in this file

REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
auth_url = REPO_URL.replace("https://", f"https://{TOKEN}@")

if not os.path.exists(REPO_DIR):
    !git clone {auth_url} {REPO_DIR}
%cd {REPO_DIR}
!git pull
!ls dataset

## Overview

In this project, you will build models that classify news articles by topic. You will represent each article with TF-IDF features (see 4) and train two classifiers introduced in this module: logistic regression and a multilayer perceptron (MLP). Transformers and pretrained language models are outside the scope of this project.

You will investigate whether models trained on one news dataset generalize to unseen in-domain articles, transfer to articles collected from a different source, and adapt when category definitions change. You will also study how regularization and limited target-domain training data affect performance.

Key concepts of this project include:
- construct a reproducible text-classification pipeline without evaluation-data leakage;
- compare a linear classifier with a nonlinear neural-network classifier;
- select hyperparameters using validation data and appropriate evaluation metrics;
- analyze in-domain performance, cross-dataset transfer, and category mismatch; and
- support quantitative results with error analysis and clear explanations.


### TODOs

[ ] Remove unnecessary (debug purpose) prints and displays

[x] Compute the training/validation accuracy for both LogRes & MLP

[ ] Compute macro-F1 for both LogRes & MLP

[ ] Compute F1 for both LogRes & MLP

[ ] Compute per-class precision for both LogRes & MLP

[ ] Compute Recall for both LogRes & MLP

[x] Refit the TF-IDF vectorizer & re-evaluate the model

[ ] Refactor training/validation loops

### Dependencies & Notebook Configurations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# for reproducibility
np.random.seed(0)
torch.manual_seed(0)

In [ ]:
# Use GPU if available
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

USE_CUDA = DEVICE.type  == "cuda"

print(f"DEVICE: {DEVICE}")
print(f"USE_CUDA: {USE_CUDA}")

## Data Load & Preprocessing

In [ ]:
# Set A
SOURCE_A_TRAIN = pd.read_csv("./dataset/dataset_a_train.csv")
SOURCE_A_TEST = pd.read_csv("./dataset/dataset_a_test.csv")

display(SOURCE_A_TRAIN.head())
display(SOURCE_A_TEST.head())

# Set B
SOURCE_B_TEST = pd.read_csv("./dataset/dataset_b_test.csv")

display(SOURCE_B_TEST.head())

# Set C
SOURCE_C_TRAIN = pd.read_csv("./dataset/dataset_c_train.csv")
SOURCE_C_TEST = pd.read_csv("./dataset/dataset_c_test.csv")

display(SOURCE_C_TRAIN.head())
display(SOURCE_C_TEST.head())

In [ ]:
LABELS = pd.DataFrame({
    "Label": range(1, 6),
    "Set A": ["Politics", "Sports", "Business", "Technology", "Entertainment"],
    "Set B": ["Politics", "Sports", "Business", "Technology", "Entertainment"],
    "Set C": ["World", "Sports", "Business", "Technology", None]
}).set_index("Label")

display(LABELS)

_Note_: Labels 2, 3, and 4 have matching meanings across all three datasets. Label 1 does not: Politics in Data A and Data B is narrower than World in Data C. A political story may reasonably belong to the World category, but World also includes international events whose main topic is not politics. Data C also has no label 5.

## Part 1: Explore Data A

In [ ]:
print(SOURCE_A_TRAIN.shape)
#7500 examples and 3 features

print(SOURCE_A_TRAIN[SOURCE_A_TRAIN["Label"] == 1].shape[0])
#1500 examples related to Politics

print(SOURCE_A_TRAIN[SOURCE_A_TRAIN["Label"] == 2].shape[0])
#1500 examples related to Sports

print(SOURCE_A_TRAIN[SOURCE_A_TRAIN["Label"] == 3].shape[0])
#1500 examples related to Business

print(SOURCE_A_TRAIN[SOURCE_A_TRAIN["Label"] == 4].shape[0])
#1500 examples related to Technology

print(SOURCE_A_TRAIN[SOURCE_A_TRAIN["Label"] == 5].shape[0])
#1500 examples related to Entertainment

SOURCE_A_TRAIN.drop_duplicates()
#Drop Duplicate Rows

SOURCE_A_TRAIN.dropna()
#Drop rows with NA values

SOURCE_A_TRAIN = SOURCE_A_TRAIN[SOURCE_A_TRAIN["Label"] != '']
SOURCE_A_TRAIN = SOURCE_A_TRAIN[SOURCE_A_TRAIN["Title"] != '']
SOURCE_A_TRAIN = SOURCE_A_TRAIN[SOURCE_A_TRAIN["Description"] != '']
#Drop rows that have empty text

SOURCE_A_TRAIN['title_tokens'] = SOURCE_A_TRAIN["Title"].str.split().str.len()
#Count Number of Tokens in Title

SOURCE_A_TRAIN['description_tokens'] = SOURCE_A_TRAIN["Description"].str.split().str.len()
#Count Number of Tokens in Description

summary = SOURCE_A_TRAIN[["title_tokens", "description_tokens"]].describe()
print(summary)
#Token distribution length for title and description

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
axes[0].boxplot(SOURCE_A_TRAIN["title_tokens"])
axes[0].set_xticklabels(["Title"])
axes[0].set_title('Token-Length Distributions for Title')
axes[0].set_ylabel("Tokens Per Example")

axes[1].boxplot(SOURCE_A_TRAIN["description_tokens"])
axes[1].set_xticklabels(["Description"])
axes[1].set_title('Token-Length Distributions for Description')
axes[1].set_ylabel("Tokens Per Example")

plt.tight_layout()
plt.show()

In [ ]:
label_names = {
    1: "Politics",
    2: "Sports",
    3: "Business",
    4: "Technology",
    5: "Entertainment"
}

for label, class_name in label_names.items():
    example = SOURCE_A_TRAIN[
        SOURCE_A_TRAIN["Label"] == label
    ].sample(n=1, random_state=40)

    print(f"\nClass {label}: {class_name}")

    print("Title: " + example["Title"].iloc[0])
    print("Description: " + example["Description"].iloc[0])

Inspect and discuss representative examples from every class.

### Class 1: Politics

**Title:** Movement Against Juvenile Court Fees Runs Into Resistance

**Description:** In almost every state and the District of Columbia, the minors who appear in the roughly one million juvenile court cases

**Analysis:** Mentions the words court and cases. Although cases can be attributed to different meanings.

### Class 2: Sports
**Title:** Lindsey Vonn Wins Bronze In What's Likely Her Last Olympic Downhill Race

**Description:** "I have no regrets; I’m really happy," she said.

**Analysis:** Mentions phrases such as the olympics and bronze.

### Class 3: Business
**Title:** 10 Habits of Ultra-Likeable Leaders

**Description:** Becoming a more likable leader is completely under your control, and it's a matter of emotional intelligence (EQ). Unlike innate, fixed characteristics, such as your intelligence (IQ), EQ is a flexible skill that you can improve with effort.

**Analysis:** Mentions phrases such as leader, intelligence, skill, and other business-like phrases.

### Class 4: Technology

**Title:** Apple Accelerates Development Of Its Electric Car

**Description:** Apple Inc. is accelerating efforts to build an electric car, designating it internally as a “committed project” and setting

**Analysis:** Mentions phrases such as Apple, an electric car, and "commited project" which are all tech related words.

### Class 5: Entertainment
**Title:** 5 Netflix Things To Watch If You Can't Wait For The Royal Wedding

**Description:** If the upcoming royal wedding is very much your thing, this is for you.

**Analysis:** Mentions the phrases "Netflix" and "Watch"


Determine whether the classes are balanced, and identify vocabulary or formatting patterns
that could act as misleading shortcuts for a classifier.

I noticed that there was a Star Wars example classified as Technology when there were phrases mentioned like "BB-8" and droid. The Star Wars example should have been classified as Entertainment and the model may use the incorrect data to classify most Star Wars-related examples as Entertainment.

There are also some words that have multiple meanings. For example, a bank could refer to the area near a river or it could mean a place where people deposit money into. The model may get confused with certain important words and classify the examples incorrectly.

### Reload untouched data
Part 1 adds columns and filters rows for exploration. Reloading here keeps every model below trained on the original files.

In [ ]:
SOURCE_A_TRAIN = pd.read_csv("./dataset/dataset_a_train.csv")
SOURCE_A_TEST = pd.read_csv("./dataset/dataset_a_test.csv")
SOURCE_B_TEST = pd.read_csv("./dataset/dataset_b_test.csv")
SOURCE_C_TRAIN = pd.read_csv("./dataset/dataset_c_train.csv")
SOURCE_C_TEST = pd.read_csv("./dataset/dataset_c_test.csv")
print(SOURCE_A_TRAIN.shape, SOURCE_A_TEST.shape, SOURCE_B_TEST.shape, SOURCE_C_TRAIN.shape, SOURCE_C_TEST.shape)

## Part 2: Build an In-Domain Baseline on Data A

### Build a TF-IDF Table

In [ ]:
# Set A
a_train, a_validation = train_test_split(
    SOURCE_A_TRAIN,
    test_size=0.2,
    random_state=0,
    stratify=SOURCE_A_TRAIN["Label"]  # label-stratified split
)

display(a_train)
display(a_validation)

In [ ]:
BATCH_SIZE = 32

In [ ]:
vectorizer = TfidfVectorizer()  # default ngram_range is (1, 1), i.e. unigram

def concat_article(row):
    return row["Title"] + ' ' + row["Description"]

a_train_tfidf = vectorizer.fit_transform(a_train.apply(concat_article, axis=1))
a_validation_tfidf = vectorizer.transform(a_validation.apply(concat_article, axis=1))

a_train_tensor = torch.tensor(a_train_tfidf.toarray(), dtype=torch.float32).to(DEVICE)
a_validation_tensor = torch.tensor(a_validation_tfidf.toarray(), dtype=torch.float32).to(DEVICE)
print(a_train_tensor)
print(a_validation_tensor)

# CrossEntropyLoss expects zero-indexed labels, so label - 1
a_train_labels_tensor = torch.tensor((a_train["Label"] - 1).to_list(), dtype=torch.long).to(DEVICE)
a_validation_labels_tensor = torch.tensor((a_validation["Label"] - 1).to_list(), dtype=torch.long).to(DEVICE)
print(a_train_labels_tensor)
print(a_validation_labels_tensor)

In [ ]:
a_train_dataset = TensorDataset(
    a_train_tensor.cpu(),
    a_train_labels_tensor.cpu()
)
a_validation_dataset = TensorDataset(
    a_validation_tensor.cpu(),
    a_validation_labels_tensor.cpu()
)

a_train_loader = DataLoader(
    a_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,  # ensure the training data is shuffled in every epoch
    pin_memory=USE_CUDA
)
a_validation_loader = DataLoader(
    a_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=USE_CUDA
)

### Logistic Regressor

In [ ]:
# Constants & Hyperparameters
MAX_EPOCH = 50  # TODO: maybe we can do some early stopping during the training process
LEARNING_RATE = 0.01

In [ ]:
model = torch.nn.Linear(a_train_tensor.shape[1], LABELS.shape[0]).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE)

In [ ]:
# TODO: Convert this into a function so we don't need to write down the whole bloody lines of code every single time we update it
training_losses = []
validation_losses = []

for epoch in range(MAX_EPOCH):
    print(f"Epoch: {epoch + 1:02}/{MAX_EPOCH}", end='')
    # Training
    model.train()

    training_loss = 0  # compute the cumulative loss within an epoch

    for i, (x, y) in enumerate(a_train_loader):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)

        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # multiply by batch size to get the total loss for this batch
        training_loss += loss.item() * x.size(0)

    training_loss /= len(a_train_dataset)  # len(dataset) == len(tensor)
    training_losses.append(training_loss)

    # Validation
    model.eval()

    validation_loss = 0

    with torch.inference_mode():
        for i, (x, y) in enumerate(a_validation_loader):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            logits = model(x)

            loss = criterion(logits, y)

            validation_loss += loss.item() * x.size(0)

    validation_loss /= len(a_validation_dataset)

    validation_losses.append(validation_loss)

    print("\tDone!")

In [ ]:
plt.figure(figsize=(10, 7))

EPOCHS = range(1, MAX_EPOCH + 1)

plt.plot(EPOCHS, training_losses, label="Training Loss")
plt.plot(EPOCHS, validation_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training & Validation Losses (Logistic Regressor)")

plt.figtext(1, 0, f"Learning Rate: {LEARNING_RATE} / Batch Size: {BATCH_SIZE}", ha="right")

plt.legend()

plt.tight_layout()
plt.show()

### Custom Multi-Layer Perceptron

The model consists of the following layers:
1. Input Layer: input_dim
2. Hidden Layer 1: 256 neurons $\to$ ReLU activation
3. Hidden Layer 2: 128 neurons $\to$ ReLU activation
4. Hidden Layer 3: 64 neurons $\to$ ReLU activation
5. Output Layer: output_dim (number of labels)

Loss Function: Cross-Entropy

Optimizer: SGD

In [ ]:
class CustomMLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.layers = nn.Sequential(
            # nn.Linear(input_dim, 256),
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            # nn.Linear(256, 128),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # nn.Linear(128, 64),
            # nn.ReLU(),
            # nn.Dropout(0.4),

            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
# Constants & Hyperparameters
MAX_EPOCH = 35

LEARNING_RATE = 0.024
MOMENTUM = 0.9
WEIGHT_DECAY = 0.001

In [ ]:
model = CustomMLP(a_train_tensor.shape[1], LABELS.shape[0]).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

In [ ]:
training_losses = []
validation_losses = []

training_accuracies = []
validation_accuracies = []

for epoch in range(MAX_EPOCH):
    print(f"Epoch: {epoch + 1:02}/{MAX_EPOCH}", end='')
    # Training
    model.train()

    training_loss = 0  # compute the cumulative loss within an epoch
    training_accuracy = 0

    for i, (x, y) in enumerate(a_train_loader):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)

        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # multiply by batch size to get the total loss for this batch
        training_loss += loss.item() * x.size(0)

        predictions = torch.argmax(logits, dim=1)
        training_accuracy += (predictions == y).sum().item()

    training_loss /= len(a_train_dataset)
    training_accuracy /= len(a_train_dataset)

    training_losses.append(training_loss)
    training_accuracies.append(training_accuracy)

    # Validation
    model.eval()

    validation_loss = 0
    validation_accuracy = 0

    with torch.inference_mode():
        for i, (x, y) in enumerate(a_validation_loader):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            logits = model(x)

            loss = criterion(logits, y)

            validation_loss += loss.item() * x.size(0)

            predictions = torch.argmax(logits, dim=1)
            validation_accuracy += (predictions == y).sum().item()

    validation_loss /= len(a_validation_dataset)
    validation_accuracy /= len(a_validation_dataset)

    validation_losses.append(validation_loss)
    validation_accuracies.append(validation_accuracy)

    print("\tDone!")

In [ ]:
plt.figure(figsize=(10, 7))

EPOCHS = range(1, MAX_EPOCH + 1)

plt.plot(EPOCHS, training_losses, label="Training Loss")
plt.plot(EPOCHS, validation_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training & Validation Losses (Custom MLP)")

plt.figtext(1, 0, f"Learning Rate: {LEARNING_RATE} / Batch Size: {BATCH_SIZE} / Momentum: {MOMENTUM}", ha="right")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

EPOCHS = range(1, MAX_EPOCH + 1)

plt.plot(EPOCHS, training_accuracies, label="Training Accuracy")
plt.plot(EPOCHS, validation_accuracies, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Training & Validation Accuracies (Custom MLP)")

plt.figtext(1, 0, f"Learning Rate: {LEARNING_RATE} / Batch Size: {BATCH_SIZE} / Momentum: {MOMENTUM}", ha="right")

plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
a_test_tfidf = vectorizer.transform(SOURCE_A_TEST.apply(concat_article, axis=1))

a_test_tensor = torch.tensor(a_test_tfidf.toarray(), dtype=torch.float32).to(DEVICE)
print(a_test_tensor)

a_test_labels_tensor = torch.tensor((SOURCE_A_TEST["Label"] - 1).to_list(), dtype=torch.long).to(DEVICE)
print(a_test_labels_tensor)

## Refit & Test

In [ ]:
a_test_dataset = TensorDataset(
    a_test_tensor.cpu(),
    a_test_labels_tensor.cpu()
)

a_test_loader = DataLoader(
    a_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=USE_CUDA
)

In [ ]:
testing_predictions = []
testing_trues = []

with torch.inference_mode():
    for i, (x, y) in enumerate(a_test_loader):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        logits = model(x)

        p = torch.argmax(logits, dim=1)

        testing_predictions.extend(p.cpu().numpy())
        testing_trues.extend(y.cpu().numpy())

testing_accuracy = accuracy_score(testing_predictions, testing_trues)

print(testing_accuracy)


## Part 3: Regularization and Transfer to Data B

In [ ]:
import copy
import json
import os

import joblib
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score)

SEED = 0              # same seed as Part 2
CLASS_IDS = [1, 2, 3, 4, 5]
NUM_CLASSES = len(CLASS_IDS)
MAX_EPOCHS = 30       # upper limit during the validation sweep
PATIENCE = 5          # early stopping patience (epochs without val macro-F1 gain)
P3_BATCH_SIZE = 64

os.makedirs("results/part3", exist_ok=True)
os.makedirs("models", exist_ok=True)


def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def article_text(df):
    # title + description, same as concat_article but safe for missing values
    return (df["Title"].fillna("").astype(str) + " " + df["Description"].fillna("").astype(str))

In [ ]:
class RegMLP(nn.Module):
    """Same architecture as the Part 2 CustomMLP (input -> 128 -> 64 -> 5, ReLU),
    with the dropout rates exposed so Part 3 can vary them."""
    def __init__(self, input_dim, output_dim, dropout=(0.2, 0.3)):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(dropout[0]),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout[1]),
            nn.Linear(64, output_dim),
        )

    def forward(self, x):
        return self.layers(x)


def build_model(kind, input_dim, dropout=(0.0, 0.0)):
    if kind == "lr":
        return nn.Linear(input_dim, NUM_CLASSES)  # multinomial logistic regression
    return RegMLP(input_dim, NUM_CLASSES, dropout)

In [ ]:
def batches(X, y=None, batch_size=P3_BATCH_SIZE, shuffle=False, rng=None):
    """Densify one batch at a time so the full TF-IDF matrix never sits on the GPU."""
    idx = np.arange(X.shape[0])
    if shuffle:
        rng.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        b = idx[start:start + batch_size]
        xb = torch.tensor(X[b].toarray(), dtype=torch.float32, device=DEVICE)
        if y is None:
            yield xb
        else:
            yield xb, torch.tensor(y[b], dtype=torch.long, device=DEVICE)


def predict(model, X):
    model.eval()
    preds = []
    with torch.inference_mode():
        for xb in batches(X):
            preds.append(model(xb).argmax(dim=1).cpu().numpy())
    return np.concatenate(preds) + 1  # back to labels 1-5


def evaluate(y_true, y_pred, labels=CLASS_IDS):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "per_class": classification_report(y_true, y_pred, labels=labels,
                                           output_dict=True, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def train_model(kind, setting, X_train, y_train, X_val=None, y_val=None,
                epochs=MAX_EPOCHS, seed=SEED):
    """Train one model. y values are the original labels 1-5.
    With validation data: early stopping on validation macro-F1, keeps the best epoch.
    Without validation data: trains for exactly `epochs` epochs (used for the refit)."""
    set_seed(seed)
    rng = np.random.default_rng(seed)
    model = build_model(kind, X_train.shape[1], setting.get("dropout", (0.0, 0.0))).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=setting["lr"],
                           weight_decay=setting["weight_decay"])
    criterion = nn.CrossEntropyLoss()
    y_train0 = np.asarray(y_train) - 1  # CrossEntropyLoss wants 0-indexed labels

    best_f1, best_epoch, best_state, bad = -1.0, 0, None, 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        for xb, yb in batches(X_train, y_train0, shuffle=True, rng=rng):
            loss = criterion(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(yb)
        row = {"epoch": epoch, "train_loss": total / X_train.shape[0]}

        if X_val is not None:
            row["val_macro_f1"] = evaluate(y_val, predict(model, X_val))["macro_f1"]
            history.append(row)
            if row["val_macro_f1"] > best_f1:
                best_f1, best_epoch, bad = row["val_macro_f1"], epoch, 0
                best_state = copy.deepcopy(model.state_dict())
            else:
                bad += 1
                if bad >= PATIENCE:
                    break
        else:
            history.append(row)

    if X_val is not None:
        model.load_state_dict(best_state)
    else:
        best_epoch = epochs
    return model, best_epoch, history

### Regularization settings
Logistic regression: L2 penalty through `weight_decay`. MLP: dropout and L2 `weight_decay`, from none to strong. Both use Adam and early stopping on Data A validation macro-F1.

In [ ]:
LR_SETTINGS = {
    "lr_wd0":    {"lr": 1e-2, "weight_decay": 0.0},
    "lr_wd1e-5": {"lr": 1e-2, "weight_decay": 1e-5},
    "lr_wd1e-4": {"lr": 1e-2, "weight_decay": 1e-4},
    "lr_wd1e-3": {"lr": 1e-2, "weight_decay": 1e-3},
}

MLP_SETTINGS = {
    "mlp_none":       {"lr": 1e-3, "weight_decay": 0.0,  "dropout": (0.0, 0.0)},
    "mlp_dropout":    {"lr": 1e-3, "weight_decay": 0.0,  "dropout": (0.2, 0.3)},
    "mlp_dropout_wd": {"lr": 1e-3, "weight_decay": 1e-4, "dropout": (0.2, 0.3)},
    "mlp_strong":     {"lr": 1e-3, "weight_decay": 1e-3, "dropout": (0.5, 0.5)},
}

ALL_SETTINGS = {**{k: ("lr", v) for k, v in LR_SETTINGS.items()},
                **{k: ("mlp", v) for k, v in MLP_SETTINGS.items()}}

### Step 1: Sweep on the Data A validation split

In [ ]:
# Same stratified 80/20 split as Part 2 (same random_state)
p3_train, p3_val = train_test_split(SOURCE_A_TRAIN, test_size=0.2,
                                    random_state=SEED, stratify=SOURCE_A_TRAIN["Label"])

p3_vec = TfidfVectorizer()  # raw unigram, fit on the 80% training split only
X_tr = p3_vec.fit_transform(article_text(p3_train))
X_val = p3_vec.transform(article_text(p3_val))
y_tr = p3_train["Label"].to_numpy()
y_val = p3_val["Label"].to_numpy()
print("train:", X_tr.shape, " val:", X_val.shape)

sweep_rows, best_epochs, histories = [], {}, {}
for name, (kind, setting) in ALL_SETTINGS.items():
    model, best_epoch, hist = train_model(kind, setting, X_tr, y_tr, X_val, y_val)
    m = evaluate(y_val, predict(model, X_val))
    best_epochs[name], histories[name] = best_epoch, hist
    sweep_rows.append({
        "setting": name, "model": kind.upper(), "lr": setting["lr"],
        "weight_decay": setting["weight_decay"], "dropout": str(setting.get("dropout", "n/a")),
        "best_epoch": best_epoch, "val_accuracy": m["accuracy"], "val_macro_f1": m["macro_f1"],
    })
    print(f"{name:16s} best_epoch={best_epoch:2d}  val acc={m['accuracy']:.4f}  val macro-F1={m['macro_f1']:.4f}")

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv("results/part3/validation_sweep.csv", index=False)
display(sweep_df)

In [ ]:
# Validation macro-F1 per epoch for every setting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, kind in zip(axes, ["lr", "mlp"]):
    for name, (k, _) in ALL_SETTINGS.items():
        if k == kind:
            h = pd.DataFrame(histories[name])
            ax.plot(h["epoch"], h["val_macro_f1"], marker="o", label=name)
    ax.set_title(f"Data A validation macro-F1 ({kind.upper()})")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Macro-F1")
    ax.legend()
plt.tight_layout()
plt.savefig("results/part3/validation_curves.png", dpi=150)
plt.show()

### Step 2: Select one setting per model (validation only)
The selection is saved before any test set is touched.

In [ ]:
best_lr = sweep_df[sweep_df.model == "LR"].sort_values("val_macro_f1", ascending=False).iloc[0]["setting"]
best_mlp = sweep_df[sweep_df.model == "MLP"].sort_values("val_macro_f1", ascending=False).iloc[0]["setting"]

selection = {
    "seed": SEED,
    "criterion": "highest Data A validation macro-F1",
    "selected_lr": best_lr, "lr_config": ALL_SETTINGS[best_lr][1], "lr_epochs": best_epochs[best_lr],
    "selected_mlp": best_mlp, "mlp_config": ALL_SETTINGS[best_mlp][1], "mlp_epochs": best_epochs[best_mlp],
}
with open("results/part3/selection.json", "w") as f:
    json.dump(selection, f, indent=2)
print(json.dumps(selection, indent=2))

### Step 3: Refit on all of Data A train, evaluate on Data A test and Data B test
The refit trains for the best epoch count found on validation, since there is no validation set left to early-stop on.

In [ ]:
vec_full = TfidfVectorizer()  # refit on ALL of dataset_a_train.csv
X_a_full = vec_full.fit_transform(article_text(SOURCE_A_TRAIN))
y_a_full = SOURCE_A_TRAIN["Label"].to_numpy()
X_a_test = vec_full.transform(article_text(SOURCE_A_TEST))
y_a_test = SOURCE_A_TEST["Label"].to_numpy()
X_b_test = vec_full.transform(article_text(SOURCE_B_TEST))  # transform only, never fit
y_b_test = SOURCE_B_TEST["Label"].to_numpy()
print("vocab size:", len(vec_full.vocabulary_))

FINAL_MODELS, final_results = {}, {}
for tag, name in [("lr", best_lr), ("mlp", best_mlp)]:
    kind, setting = ALL_SETTINGS[name]
    model, _, _ = train_model(kind, setting, X_a_full, y_a_full, epochs=best_epochs[name])
    FINAL_MODELS[tag] = model
    torch.save(model.state_dict(), f"models/part3_final_{tag}.pt")
    final_results[tag] = {
        "setting": name, "config": setting, "epochs": best_epochs[name], "seed": SEED,
        "representation": "unigram TF-IDF, title + description",
        "train_data": "all of dataset_a_train.csv", "train_size": len(SOURCE_A_TRAIN),
        "A_test": evaluate(y_a_test, predict(model, X_a_test)),
        "B_test": evaluate(y_b_test, predict(model, X_b_test)),
    }
joblib.dump(vec_full, "models/part3_vectorizer.joblib")

with open("results/part3/final_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

summary = pd.DataFrame([{
    "model": tag.upper(), "setting": r["setting"],
    "A_test_acc": r["A_test"]["accuracy"], "A_test_macro_f1": r["A_test"]["macro_f1"],
    "B_test_acc": r["B_test"]["accuracy"], "B_test_macro_f1": r["B_test"]["macro_f1"],
} for tag, r in final_results.items()])
display(summary)

In [ ]:
# Per-class precision / recall / F1, Data A vs Data B
rows = []
for tag, r in final_results.items():
    for lab in CLASS_IDS:
        a, b = r["A_test"]["per_class"][str(lab)], r["B_test"]["per_class"][str(lab)]
        rows.append({"model": tag.upper(), "label": lab, "class": LABELS.loc[lab, "Set A"],
                     "A_precision": a["precision"], "A_recall": a["recall"], "A_f1": a["f1-score"],
                     "B_precision": b["precision"], "B_recall": b["recall"], "B_f1": b["f1-score"],
                     "f1_drop": a["f1-score"] - b["f1-score"]})
per_class = pd.DataFrame(rows)
per_class.to_csv("results/part3/per_class_A_vs_B.csv", index=False)
display(per_class.round(3))

# Confusion matrices
class_names = list(LABELS["Set A"])
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for i, tag in enumerate(["lr", "mlp"]):
    for j, ds in enumerate(["A_test", "B_test"]):
        cm = np.array(final_results[tag][ds]["confusion_matrix"])
        ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
            ax=axes[i, j], colorbar=False, xticks_rotation=45)
        axes[i, j].set_title(f"{tag.upper()} on {ds.replace('_', ' ')}")
plt.tight_layout()
plt.savefig("results/part3/confusion_matrices.png", dpi=150)
plt.show()

### Step 4 (post hoc): every setting on the test sets
Reported for analysis only. These numbers were not used to pick the final models.

In [ ]:
posthoc = []
for name, (kind, setting) in ALL_SETTINGS.items():
    model, _, _ = train_model(kind, setting, X_a_full, y_a_full, epochs=best_epochs[name])
    fa = evaluate(y_a_test, predict(model, X_a_test))["macro_f1"]
    fb = evaluate(y_b_test, predict(model, X_b_test))["macro_f1"]
    posthoc.append({"setting": name, "model": kind.upper(),
                    "A_test_macro_f1": fa, "B_test_macro_f1": fb, "A_minus_B": fa - fb})
posthoc_df = pd.DataFrame(posthoc)
posthoc_df.to_csv("results/part3/posthoc_all_settings.csv", index=False)
display(posthoc_df.round(4))

### Step 5: Transfer diagnostics (for the A vs B discussion)
Class balance, text length in words, and out-of-vocabulary rate relative to the Data A vocabulary.

In [ ]:
def word_count(s):
    return s.fillna("").astype(str).str.split().str.len()

analyzer, vocab = vec_full.build_analyzer(), vec_full.vocabulary_
diag_rows = []
for dname, df in [("A_train", SOURCE_A_TRAIN), ("A_test", SOURCE_A_TEST), ("B_test", SOURCE_B_TEST)]:
    toks = [t for doc in article_text(df) for t in analyzer(doc)]
    props = df["Label"].value_counts(normalize=True).sort_index()
    diag_rows.append({
        "dataset": dname, "n": len(df),
        **{f"pct_label_{k}": round(100 * props.get(k, 0), 1) for k in CLASS_IDS},
        "title_words_median": word_count(df["Title"]).median(),
        "desc_words_median": word_count(df["Description"]).median(),
        "oov_rate_vs_A_vocab": round(float(np.mean([t not in vocab for t in toks])), 4),
    })
diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv("results/part3/transfer_diagnostics.csv", index=False)
display(diag_df)

### Frozen models for Part 4
`FINAL_MODELS["lr"]`, `FINAL_MODELS["mlp"]`, and `vec_full` stay in memory. They are also saved to `models/`. To reload them in a fresh session:

```python
vec_full = joblib.load("models/part3_vectorizer.joblib")
n_feat = len(vec_full.vocabulary_)
FINAL_MODELS = {}
for tag in ["lr", "mlp"]:
    kind, setting = ALL_SETTINGS[selection[f"selected_{tag}"]]
    m = build_model(kind, n_feat, setting.get("dropout", (0.0, 0.0))).to(DEVICE)
    m.load_state_dict(torch.load(f"models/part3_final_{tag}.pt", map_location=DEVICE))
    FINAL_MODELS[tag] = m.eval()
```

### Duplicate check
Checking whether duplicate articles explain the gap between validation and A test scores.

In [ ]:
dup_rows = SOURCE_A_TRAIN.duplicated().sum()
dup_text = article_text(SOURCE_A_TRAIN).duplicated().sum()
train_text = set(article_text(p3_train))
val_overlap = article_text(p3_val).isin(train_text).sum()
test_overlap = article_text(SOURCE_A_TEST).isin(set(article_text(SOURCE_A_TRAIN))).sum()
print("duplicate rows in A train:", dup_rows)
print("duplicate texts in A train:", dup_text)
print("val articles also in train split:", val_overlap, "/", len(p3_val))
print("A test articles also in A train:", test_overlap, "/", len(SOURCE_A_TEST))

## Part 4: Transfer to Data C and Analysis of Label 1
The two frozen Part 3 models (`FINAL_MODELS`) and the Data A vectorizer (`vec_full`) are applied to `dataset_c_test.csv` with `transform` only. Nothing is refit on Data A, and `dataset_c_test.csv` is never used for training or selection.

In [ ]:
from collections import Counter
from sklearn.metrics import precision_recall_fscore_support

os.makedirs("results/part4", exist_ok=True)
C_TRAIN = pd.read_csv("./dataset/dataset_c_train.csv")
C_TEST = pd.read_csv("./dataset/dataset_c_test.csv")
y_c = C_TEST["Label"].to_numpy()
print("C train:", C_TRAIN.shape, " C test:", C_TEST.shape, C_TEST["Label"].value_counts().sort_index().to_dict())

X_c = vec_full.transform(article_text(C_TEST))   # frozen Data A vocabulary
pred_c = {tag: predict(m, X_c) for tag, m in FINAL_MODELS.items()}   # predictions use labels 1-5

### Step 1: Shared-label transfer (2-4), semantic transfer (label 1), and label 5 predictions
Labels 2-4 mean the same thing in A and C, so they are scored directly on the rows whose true label is 2, 3 or 4. Label 1 is Politics in A but World in C, so it is reported separately.

In [ ]:
shared = np.isin(y_c, [2, 3, 4])
rows = []
for tag, yp in pred_c.items():
    p, r, f, _ = precision_recall_fscore_support(y_c, yp, labels=[1], zero_division=0)
    rows.append({
        "model": tag.upper(), "setting": selection[f"selected_{tag}"],
        "shared_acc_2to4": accuracy_score(y_c[shared], yp[shared]),
        "shared_macro_f1_2to4": f1_score(y_c[shared], yp[shared], labels=[2, 3, 4], average="macro", zero_division=0),
        "label1_precision": p[0], "label1_recall": r[0], "label1_f1": f[0],
        "pct_pred_label5": 100 * np.mean(yp == 5),
    })
transfer_df = pd.DataFrame(rows)
transfer_df.to_csv("results/part4/transfer_summary.csv", index=False)
display(transfer_df.round(4))

# Where do the true World (label 1) articles go?
world_pred = pd.DataFrame({tag.upper(): pd.Series(yp[y_c == 1]).value_counts().reindex(CLASS_IDS, fill_value=0)
                           for tag, yp in pred_c.items()})
world_pred.index.name = "predicted label (true label 1 = World)"
display(world_pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (tag, yp) in zip(axes, pred_c.items()):
    cm = confusion_matrix(y_c, yp, labels=CLASS_IDS)[:4]   # rows: true 1-4, cols: predicted 1-5
    ax.imshow(cm, cmap="Blues")
    for i in range(4):
        for j in range(5):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    ax.set_xticks(range(5), ["1 Pol/World", "2 Sports", "3 Biz", "4 Tech", "5 Ent"], rotation=30)
    ax.set_yticks(range(4), ["1 World", "2 Sports", "3 Biz", "4 Tech"])
    ax.set_title(f"{tag.upper()} (Data A model) on Data C test"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.savefig("results/part4/confusion_matrices.png", dpi=150); plt.show()

### Step 2: Errors on true label 1 (World)
A sample of misclassified World articles per model, then words that appear in World articles missed by both models but never appear in Data A Politics training articles.

In [ ]:
pd.set_option("display.max_colwidth", 200)
for tag, yp in pred_c.items():
    miss = (y_c == 1) & (yp != 1)
    errs = C_TEST[miss].assign(Predicted=yp[miss])
    print(f"{tag.upper()}: {miss.sum()} of {(y_c == 1).sum()} World articles misclassified")
    sample = errs.sample(12, random_state=SEED)[["Predicted", "Title", "Description"]]
    sample.to_csv(f"results/part4/label1_errors_{tag}.csv")
    display(sample)

analyzer = vec_full.build_analyzer()
politics_vocab = {t for doc in article_text(SOURCE_A_TRAIN[SOURCE_A_TRAIN["Label"] == 1]) for t in analyzer(doc)}
missed_both = (y_c == 1) & (pred_c["lr"] != 1) & (pred_c["mlp"] != 1)
counts = Counter(t for doc in article_text(C_TEST[missed_both]) for t in set(analyzer(doc)) if t not in politics_vocab)
print(f"{missed_both.sum()} World articles missed by both models. Most common words never seen in Data A Politics:")
display(pd.DataFrame(counts.most_common(25), columns=["word", "n_missed_articles"]))

### Step 3: In-domain reference models trained on `dataset_c_train.csv`
Same selected Part 3 settings. The number of epochs is chosen by early stopping on an internal 80/20 split of Data C train, then the vectorizer and model are refit on all of Data C train with 4 output classes and evaluated once on Data C test. These are kept separate from the transferred models.

In [ ]:
c_tr, c_val = train_test_split(C_TRAIN, test_size=0.2, random_state=SEED, stratify=C_TRAIN["Label"])
vec_split = TfidfVectorizer()
Xc_tr = vec_split.fit_transform(article_text(c_tr))
Xc_val = vec_split.transform(article_text(c_val))

vec_c = TfidfVectorizer()   # refit on all of Data C train
Xc_full = vec_c.fit_transform(article_text(C_TRAIN))
Xc_test = vec_c.transform(article_text(C_TEST))

NUM_CLASSES = 4   # Data C has labels 1-4 only
ref_pred = {}
for tag in ["lr", "mlp"]:
    kind, setting = ALL_SETTINGS[selection[f"selected_{tag}"]]
    _, ep, _ = train_model(kind, setting, Xc_tr, c_tr["Label"].to_numpy(), Xc_val, c_val["Label"].to_numpy())
    model, _, _ = train_model(kind, setting, Xc_full, C_TRAIN["Label"].to_numpy(), epochs=ep)
    ref_pred[tag] = predict(model, Xc_test)
    print(f"{tag.upper()} reference: setting={selection[f'selected_{tag}']}, epochs={ep}, seed={SEED}, train size={len(C_TRAIN)}")
NUM_CLASSES = 5   # restore for the Data A models

cmp = []
for tag in ["lr", "mlp"]:
    for source, yp in [("A-trained (transfer)", pred_c[tag]), ("C-trained (reference)", ref_pred[tag])]:
        p, r, f, _ = precision_recall_fscore_support(y_c, yp, labels=[1], zero_division=0)
        cmp.append({"model": tag.upper(), "source": source, "label1_recall": r[0], "label1_f1": f[0],
                    "macro_f1_1to4": f1_score(y_c, yp, labels=[1, 2, 3, 4], average="macro", zero_division=0)})
compare_df = pd.DataFrame(cmp)
compare_df.to_csv("results/part4/transfer_vs_reference.csv", index=False)
display(compare_df.round(4))

## Part 5: Adding Limited Data C Training Examples
Uses the final settings chosen in Part 3 (no Data C results were used to pick them). Only `dataset_c_train.csv` is ever added to training. `dataset_c_test.csv` is evaluation only.

In [ ]:
SOURCE_C_TRAIN = pd.read_csv("./dataset/dataset_c_train.csv")
SOURCE_C_TEST = pd.read_csv("./dataset/dataset_c_test.csv")
print("C train", SOURCE_C_TRAIN.shape, " C test", SOURCE_C_TEST.shape)
print("\nC train class counts:")
print(SOURCE_C_TRAIN["Label"].value_counts().sort_index())

### Final settings from Part 3
Loaded from `results/part3/selection.json`. Each model is refit for the same number of epochs Part 3 used.

In [ ]:
os.makedirs("results/part5", exist_ok=True)

try:
    with open("results/part3/selection.json") as f:
        selection = json.load(f)
except FileNotFoundError:
    # Values from the Part 3 run, in case the file is missing
    selection = {"selected_lr": "lr_wd1e-5", "lr_config": {"lr": 0.01, "weight_decay": 1e-05}, "lr_epochs": 6,
                 "selected_mlp": "mlp_dropout",
                 "mlp_config": {"lr": 0.001, "weight_decay": 0.0, "dropout": [0.2, 0.3]}, "mlp_epochs": 2}

FINAL = {
    "lr":  {"kind": "lr", "name": selection["selected_lr"], "setting": selection["lr_config"],
            "epochs": selection["lr_epochs"]},
    "mlp": {"kind": "mlp", "name": selection["selected_mlp"],
            "setting": {**selection["mlp_config"], "dropout": tuple(selection["mlp_config"]["dropout"])},
            "epochs": selection["mlp_epochs"]},
}
for tag, cfg in FINAL.items():
    print(tag, cfg)

### Sampling procedure
Each Data C class is shuffled once with a fixed seed. A sample takes the first *n* rows of each shuffled class, so every smaller sample is a subset of the larger ones. Budgets are split as evenly as possible across the strategy's labels.

In [ ]:
BUDGETS = [50, 150, 600, 1500]
STRATEGIES = {
    "label1_only": [1],           # Data C World examples only
    "shared_only": [2, 3, 4],     # Sports, Business, Technology
    "balanced":    [1, 2, 3, 4],  # roughly equal from all four
}
SAMPLE_SEED = 0
C_LABELS = [1, 2, 3, 4]

C_POOLS = {lab: SOURCE_C_TRAIN[SOURCE_C_TRAIN["Label"] == lab].sample(frac=1, random_state=SAMPLE_SEED)
           for lab in C_LABELS}


def per_class_counts(budget, labels):
    base, extra = divmod(budget, len(labels))
    return {lab: base + (1 if i < extra else 0) for i, lab in enumerate(labels)}


def sample_c(budget, strategy):
    counts = per_class_counts(budget, STRATEGIES[strategy])
    parts = []
    for lab, n in counts.items():
        if n > len(C_POOLS[lab]):
            raise ValueError(f"Need {n} examples of C label {lab}, only {len(C_POOLS[lab])} available")
        parts.append(C_POOLS[lab].iloc[:n])
    return pd.concat(parts), counts


for s in STRATEGIES:
    print(s, {b: per_class_counts(b, STRATEGIES[s]) for b in BUDGETS})

### Metrics
- C label-1 recall and F1
- C overall macro-F1 over labels 1-4
- C strict macro-F1 over labels 2-4, computed on the full C test set (World articles predicted as Sports/Business/Tech count as false positives)
- C shared-only macro-F1 over labels 2-4, computed only on C test articles whose true label is 2-4 (same as Part 4)
- % of C test predicted as label 5 (a class C does not have)
- A test and B test macro-F1 (retained performance)

In [ ]:
TXT_A_TEST, TXT_B_TEST, TXT_C_TEST = (article_text(SOURCE_A_TEST), article_text(SOURCE_B_TEST),
                                      article_text(SOURCE_C_TEST))
y_a_test = SOURCE_A_TEST["Label"].to_numpy()
y_b_test = SOURCE_B_TEST["Label"].to_numpy()
y_c_test = SOURCE_C_TEST["Label"].to_numpy()
SHARED_MASK = np.isin(y_c_test, [2, 3, 4])


def part5_metrics(model, vec):
    pc = predict(model, vec.transform(TXT_C_TEST))
    pa = predict(model, vec.transform(TXT_A_TEST))
    pb = predict(model, vec.transform(TXT_B_TEST))
    rep = classification_report(y_c_test, pc, labels=C_LABELS, output_dict=True, zero_division=0)
    f1 = lambda yt, yp, labs: float(f1_score(yt, yp, labels=labs, average="macro", zero_division=0))
    metrics = {
        "C_label1_recall": rep["1"]["recall"],
        "C_label1_f1": rep["1"]["f1-score"],
        "C_macro_f1_1to4": f1(y_c_test, pc, C_LABELS),
        "C_strict_macro_f1_2to4": f1(y_c_test, pc, [2, 3, 4]),
        "C_shared_only_macro_f1_2to4": f1(y_c_test[SHARED_MASK], pc[SHARED_MASK], [2, 3, 4]),
        "C_pct_pred_label5": float(100 * np.mean(pc == 5)),
        "A_test_macro_f1": f1(y_a_test, pa, CLASS_IDS),
        "B_test_macro_f1": f1(y_b_test, pb, CLASS_IDS),
    }
    cm = confusion_matrix(y_c_test, pc, labels=CLASS_IDS).tolist()
    return metrics, cm

### Run every condition
Budget 0 is the frozen Part 3 model (run once per model). Every other condition refits TF-IDF and the classifier on Data A train plus the sampled Data C examples. 2 models x 3 strategies x 4 budgets = 24 runs.

In [ ]:
rows, conf_mats = [], {}

for tag, cfg in FINAL.items():
    # ---- budget 0: the Part 3 model, loaded from disk if available
    try:
        vec0 = joblib.load("models/part3_vectorizer.joblib")
        model0 = build_model(cfg["kind"], len(vec0.vocabulary_), cfg["setting"].get("dropout", (0.0, 0.0))).to(DEVICE)
        model0.load_state_dict(torch.load(f"models/part3_final_{tag}.pt", map_location=DEVICE))
        src = "loaded Part 3 model"
    except FileNotFoundError:
        vec0 = TfidfVectorizer()
        X0 = vec0.fit_transform(article_text(SOURCE_A_TRAIN))
        model0, _, _ = train_model(cfg["kind"], cfg["setting"], X0, SOURCE_A_TRAIN["Label"].to_numpy(),
                                   epochs=cfg["epochs"])
        src = "retrained Part 3 config"
    met, cm = part5_metrics(model0, vec0)
    rows.append({"model": tag.upper(), "strategy": "baseline", "budget": 0, "c_counts": "{}",
                 "train_size": len(SOURCE_A_TRAIN), "vocab_size": len(vec0.vocabulary_), **met})
    conf_mats[f"{tag}_baseline_0"] = cm
    print(f"{tag.upper()} baseline ({src}): C macro-F1={met['C_macro_f1_1to4']:.4f}  C label-1 F1={met['C_label1_f1']:.4f}")

    # ---- nonzero budgets
    for strategy in STRATEGIES:
        for budget in BUDGETS:
            c_sample, counts = sample_c(budget, strategy)
            train_df = pd.concat([SOURCE_A_TRAIN, c_sample], ignore_index=True)
            vec = TfidfVectorizer()  # refit on A train + selected C examples
            X = vec.fit_transform(article_text(train_df))
            y = train_df["Label"].to_numpy()  # C World examples keep label 1
            model, _, _ = train_model(cfg["kind"], cfg["setting"], X, y, epochs=cfg["epochs"])
            met, cm = part5_metrics(model, vec)
            rows.append({"model": tag.upper(), "strategy": strategy, "budget": budget,
                         "c_counts": str(counts), "train_size": len(train_df),
                         "vocab_size": len(vec.vocabulary_), **met})
            conf_mats[f"{tag}_{strategy}_{budget}"] = cm
            print(f"{tag.upper()} {strategy:12s} {budget:5d}  C macro-F1={met['C_macro_f1_1to4']:.4f}  "
                  f"C label-1 F1={met['C_label1_f1']:.4f}  A={met['A_test_macro_f1']:.4f}  B={met['B_test_macro_f1']:.4f}")

results_df = pd.DataFrame(rows)
results_df.to_csv("results/part5/all_conditions.csv", index=False)
with open("results/part5/confusion_matrices.json", "w") as f:
    json.dump(conf_mats, f)
with open("results/part5/config.json", "w") as f:
    json.dump({"seed": SEED, "sample_seed": SAMPLE_SEED, "budgets": BUDGETS, "strategies": STRATEGIES,
               "final_settings": {k: {**v, "setting": dict(v["setting"])} for k, v in FINAL.items()},
               "representation": "unigram TF-IDF, title + description"}, f, indent=2, default=str)

### Results tables

In [ ]:
METRIC_COLS = ["C_label1_recall", "C_label1_f1", "C_macro_f1_1to4", "C_strict_macro_f1_2to4",
               "C_shared_only_macro_f1_2to4", "C_pct_pred_label5", "A_test_macro_f1", "B_test_macro_f1"]
for tag in ["LR", "MLP"]:
    print(f"\n===== {tag} =====")
    display(results_df[results_df.model == tag][["strategy", "budget", "train_size"] + METRIC_COLS]
            .round(4).reset_index(drop=True))

### Plots: added Data C examples vs C label-1 F1 and C macro-F1

In [ ]:
TICKS = [0] + BUDGETS
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for i, tag in enumerate(["LR", "MLP"]):
    d = results_df[results_df.model == tag]
    base = d[d.budget == 0].iloc[0]
    for j, (metric, label) in enumerate([("C_label1_f1", "Data C label-1 (World) F1"),
                                         ("C_macro_f1_1to4", "Data C macro-F1 (labels 1-4)")]):
        ax = axes[i, j]
        for s in STRATEGIES:
            ds = d[d.strategy == s].sort_values("budget")
            ax.plot([0] + ds.budget.tolist(), [base[metric]] + ds[metric].tolist(), marker="o", label=s)
        ax.set_xscale("symlog", linthresh=50)
        ax.set_xticks(TICKS)
        ax.set_xticklabels(TICKS)
        ax.set_xlabel("Added Data C training examples")
        ax.set_ylabel(label)
        ax.set_title(f"{tag}: {label}")
        ax.grid(alpha=0.3)
        ax.legend()
plt.tight_layout()
plt.savefig("results/part5/budget_curves.png", dpi=150)
plt.show()

In [ ]:
# Retained performance on Data A and Data B
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, tag in enumerate(["LR", "MLP"]):
    d = results_df[results_df.model == tag]
    base = d[d.budget == 0].iloc[0]
    ax = axes[i]
    for s in STRATEGIES:
        ds = d[d.strategy == s].sort_values("budget")
        x = [0] + ds.budget.tolist()
        ax.plot(x, [base["A_test_macro_f1"]] + ds["A_test_macro_f1"].tolist(), marker="o", label=f"{s} (A)")
        ax.plot(x, [base["B_test_macro_f1"]] + ds["B_test_macro_f1"].tolist(), marker="s", linestyle="--", label=f"{s} (B)")
    ax.set_xscale("symlog", linthresh=50)
    ax.set_xticks(TICKS)
    ax.set_xticklabels(TICKS)
    ax.set_xlabel("Added Data C training examples")
    ax.set_ylabel("Macro-F1")
    ax.set_title(f"{tag}: retained Data A / Data B macro-F1")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("results/part5/retention_curves.png", dpi=150)
plt.show()

### Requirement check
Every adapted Part 5 model needs at least 40% macro-F1 on the Data C test set (labels 1-4).

In [ ]:
adapted = results_df[results_df.budget > 0]
check = adapted[["model", "strategy", "budget", "C_macro_f1_1to4"]].copy()
check["passes_40pct"] = check["C_macro_f1_1to4"] >= 0.40
display(check.round(4))
print("All adapted models pass:", bool(check["passes_40pct"].all()))

## Selected Component: Unigram + Bigram TF-IDF
**Variation:** `TfidfVectorizer(ngram_range=(1, 2), min_df=2)`. Features are single words plus two-word phrases. `min_df=2` drops any term that appears in only one training document, which keeps the much larger bigram vocabulary manageable.

**Why it might help:** phrases like "prime minister", "stock market", or "world cup" carry topic meaning that the separate words do not. That could sharpen class boundaries in domain and help separate World from Business and Technology in Data C.

**Why it might hurt:** many bigrams are specific to one source's writing style or headline habits, so they can fit Data A closely and transfer worse to Data B and Data C. The feature space also grows a lot, which makes overfitting easier for the MLP.

Everything else is identical to the required component: same split, seeds, regularization settings, selection rule, refit procedure, sampling, and metrics.

In [ ]:
EXT_REPR = "unigram+bigram TF-IDF (ngram_range=(1,2), min_df=2), title + description"


def make_ext_vec():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=2)


os.makedirs("results/extension", exist_ok=True)

### Extension, Part 3 redo: sweep on the same Data A validation split

In [ ]:
ext_vec_split = make_ext_vec()  # fit on the same 80% training split as Part 3
Xe_tr = ext_vec_split.fit_transform(article_text(p3_train))
Xe_val = ext_vec_split.transform(article_text(p3_val))
print("features  baseline:", X_tr.shape[1], "  extension:", Xe_tr.shape[1])

ext_sweep_rows, ext_best_epochs, ext_histories = [], {}, {}
for name, (kind, setting) in ALL_SETTINGS.items():
    model, best_epoch, hist = train_model(kind, setting, Xe_tr, y_tr, Xe_val, y_val)
    m = evaluate(y_val, predict(model, Xe_val))
    ext_best_epochs[name], ext_histories[name] = best_epoch, hist
    ext_sweep_rows.append({
        "setting": name, "model": kind.upper(), "lr": setting["lr"],
        "weight_decay": setting["weight_decay"], "dropout": str(setting.get("dropout", "n/a")),
        "best_epoch": best_epoch, "val_accuracy": m["accuracy"], "val_macro_f1": m["macro_f1"],
    })
    print(f"{name:16s} best_epoch={best_epoch:2d}  val acc={m['accuracy']:.4f}  val macro-F1={m['macro_f1']:.4f}")

ext_sweep_df = pd.DataFrame(ext_sweep_rows)
ext_sweep_df.to_csv("results/extension/validation_sweep.csv", index=False)
display(ext_sweep_df)

In [ ]:
# Selection by Data A validation macro-F1 only, saved before any test evaluation
ext_best_lr = ext_sweep_df[ext_sweep_df.model == "LR"].sort_values("val_macro_f1", ascending=False).iloc[0]["setting"]
ext_best_mlp = ext_sweep_df[ext_sweep_df.model == "MLP"].sort_values("val_macro_f1", ascending=False).iloc[0]["setting"]

ext_selection = {
    "seed": SEED, "representation": EXT_REPR,
    "criterion": "highest Data A validation macro-F1",
    "selected_lr": ext_best_lr, "lr_config": ALL_SETTINGS[ext_best_lr][1], "lr_epochs": ext_best_epochs[ext_best_lr],
    "selected_mlp": ext_best_mlp, "mlp_config": ALL_SETTINGS[ext_best_mlp][1], "mlp_epochs": ext_best_epochs[ext_best_mlp],
}
with open("results/extension/selection.json", "w") as f:
    json.dump(ext_selection, f, indent=2)
print(json.dumps(ext_selection, indent=2))

### Extension, Part 3 redo: refit on all of Data A train, evaluate on A test and B test

In [ ]:
ext_vec_full = make_ext_vec()
Xe_a_full = ext_vec_full.fit_transform(article_text(SOURCE_A_TRAIN))
Xe_a_test = ext_vec_full.transform(article_text(SOURCE_A_TEST))
Xe_b_test = ext_vec_full.transform(article_text(SOURCE_B_TEST))  # transform only
print("vocab size  baseline:", len(vec_full.vocabulary_), "  extension:", len(ext_vec_full.vocabulary_))

EXT_FINAL_MODELS, ext_final_results = {}, {}
for tag, name in [("lr", ext_best_lr), ("mlp", ext_best_mlp)]:
    kind, setting = ALL_SETTINGS[name]
    model, _, _ = train_model(kind, setting, Xe_a_full, y_a_full, epochs=ext_best_epochs[name])
    EXT_FINAL_MODELS[tag] = model
    torch.save(model.state_dict(), f"models/ext_final_{tag}.pt")
    ext_final_results[tag] = {
        "setting": name, "config": setting, "epochs": ext_best_epochs[name], "seed": SEED,
        "representation": EXT_REPR, "train_data": "all of dataset_a_train.csv",
        "train_size": len(SOURCE_A_TRAIN), "vocab_size": len(ext_vec_full.vocabulary_),
        "A_test": evaluate(y_a_test, predict(model, Xe_a_test)),
        "B_test": evaluate(y_b_test, predict(model, Xe_b_test)),
    }
joblib.dump(ext_vec_full, "models/ext_vectorizer.joblib")
with open("results/extension/final_results.json", "w") as f:
    json.dump(ext_final_results, f, indent=2)

# Side-by-side with the required-component Part 3 models
cmp_rows = []
for tag in ["lr", "mlp"]:
    for rep, res, vsize in [("unigram (required)", final_results, len(vec_full.vocabulary_)),
                            ("uni+bigram (extension)", ext_final_results, len(ext_vec_full.vocabulary_))]:
        r = res[tag]
        cmp_rows.append({"model": tag.upper(), "representation": rep, "setting": r["setting"],
                         "epochs": r["epochs"], "features": vsize,
                         "A_test_acc": r["A_test"]["accuracy"], "A_test_macro_f1": r["A_test"]["macro_f1"],
                         "B_test_acc": r["B_test"]["accuracy"], "B_test_macro_f1": r["B_test"]["macro_f1"]})
p3_compare = pd.DataFrame(cmp_rows)
p3_compare.to_csv("results/extension/part3_comparison.csv", index=False)
display(p3_compare.round(4))
print("Extension Part 3 models pass 50% on A and B:",
      all(ext_final_results[t][d]["macro_f1"] >= 0.50 for t in ["lr", "mlp"] for d in ["A_test", "B_test"]))

In [ ]:
# Per-class F1, required vs extension, on A test and B test
rows = []
for tag in ["lr", "mlp"]:
    for lab in CLASS_IDS:
        row = {"model": tag.upper(), "label": lab, "class": LABELS.loc[lab, "Set A"]}
        for ds in ["A_test", "B_test"]:
            base = final_results[tag][ds]["per_class"][str(lab)]["f1-score"]
            ext = ext_final_results[tag][ds]["per_class"][str(lab)]["f1-score"]
            row[f"{ds}_f1_unigram"], row[f"{ds}_f1_bigram"], row[f"{ds}_f1_change"] = base, ext, ext - base
        rows.append(row)
p3_class_compare = pd.DataFrame(rows)
p3_class_compare.to_csv("results/extension/part3_per_class_comparison.csv", index=False)
display(p3_class_compare.round(3))

# Extension confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for i, tag in enumerate(["lr", "mlp"]):
    for j, ds in enumerate(["A_test", "B_test"]):
        cm = np.array(ext_final_results[tag][ds]["confusion_matrix"])
        ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=axes[i, j], colorbar=False, xticks_rotation=45)
        axes[i, j].set_title(f"Extension {tag.upper()} on {ds.replace('_', ' ')}")
plt.tight_layout()
plt.savefig("results/extension/confusion_matrices.png", dpi=150)
plt.show()

In [ ]:
# Error-type changes: (extension confusion matrix) - (required confusion matrix)
# Off-diagonal positive = the extension makes more of that mistake. Negative = fewer.
for tag in ["lr", "mlp"]:
    for ds in ["A_test", "B_test"]:
        diff = (np.array(ext_final_results[tag][ds]["confusion_matrix"])
                - np.array(final_results[tag][ds]["confusion_matrix"]))
        print(f"\n{tag.upper()} {ds}: rows = true class, columns = predicted class")
        display(pd.DataFrame(diff, index=class_names, columns=class_names))

### Extension, Part 3 redo (post hoc): every setting on the test sets
Analysis only. Not used for selection.

In [ ]:
ext_posthoc = []
for name, (kind, setting) in ALL_SETTINGS.items():
    model, _, _ = train_model(kind, setting, Xe_a_full, y_a_full, epochs=ext_best_epochs[name])
    fa = evaluate(y_a_test, predict(model, Xe_a_test))["macro_f1"]
    fb = evaluate(y_b_test, predict(model, Xe_b_test))["macro_f1"]
    ext_posthoc.append({"setting": name, "model": kind.upper(),
                        "A_test_macro_f1": fa, "B_test_macro_f1": fb, "A_minus_B": fa - fb})
ext_posthoc_df = pd.DataFrame(ext_posthoc)
ext_posthoc_df.to_csv("results/extension/posthoc_all_settings.csv", index=False)
display(ext_posthoc_df.round(4))

### Extension, Part 5 redo: add limited Data C examples
Same budgets, strategies, sampling procedure, and seeds as Part 5. Budget 0 is the extension's Part 3 model.

In [ ]:
EXT_FINAL = {
    "lr":  {"kind": "lr", "name": ext_best_lr, "setting": ALL_SETTINGS[ext_best_lr][1],
            "epochs": ext_best_epochs[ext_best_lr]},
    "mlp": {"kind": "mlp", "name": ext_best_mlp, "setting": ALL_SETTINGS[ext_best_mlp][1],
            "epochs": ext_best_epochs[ext_best_mlp]},
}

ext_rows, ext_conf_mats = [], {}
for tag, cfg in EXT_FINAL.items():
    met, cm = part5_metrics(EXT_FINAL_MODELS[tag], ext_vec_full)
    ext_rows.append({"model": tag.upper(), "strategy": "baseline", "budget": 0, "c_counts": "{}",
                     "train_size": len(SOURCE_A_TRAIN), "vocab_size": len(ext_vec_full.vocabulary_), **met})
    ext_conf_mats[f"{tag}_baseline_0"] = cm
    print(f"{tag.upper()} baseline: C macro-F1={met['C_macro_f1_1to4']:.4f}  C label-1 F1={met['C_label1_f1']:.4f}")

    for strategy in STRATEGIES:
        for budget in BUDGETS:
            c_sample, counts = sample_c(budget, strategy)
            train_df = pd.concat([SOURCE_A_TRAIN, c_sample], ignore_index=True)
            vec = make_ext_vec()  # refit on A train + selected C examples
            X = vec.fit_transform(article_text(train_df))
            model, _, _ = train_model(cfg["kind"], cfg["setting"], X, train_df["Label"].to_numpy(),
                                      epochs=cfg["epochs"])
            met, cm = part5_metrics(model, vec)
            ext_rows.append({"model": tag.upper(), "strategy": strategy, "budget": budget,
                             "c_counts": str(counts), "train_size": len(train_df),
                             "vocab_size": len(vec.vocabulary_), **met})
            ext_conf_mats[f"{tag}_{strategy}_{budget}"] = cm
            print(f"{tag.upper()} {strategy:12s} {budget:5d}  C macro-F1={met['C_macro_f1_1to4']:.4f}  "
                  f"C label-1 F1={met['C_label1_f1']:.4f}  A={met['A_test_macro_f1']:.4f}  B={met['B_test_macro_f1']:.4f}")

ext_results_df = pd.DataFrame(ext_rows)
ext_results_df.to_csv("results/extension/part5_all_conditions.csv", index=False)
with open("results/extension/part5_confusion_matrices.json", "w") as f:
    json.dump(ext_conf_mats, f)

for tag in ["LR", "MLP"]:
    print(f"\n===== Extension {tag} =====")
    display(ext_results_df[ext_results_df.model == tag][["strategy", "budget", "train_size", "vocab_size"] + METRIC_COLS]
            .round(4).reset_index(drop=True))

ext_check = ext_results_df[ext_results_df.budget > 0]
print("All extension adapted models pass 40% C macro-F1:", bool((ext_check["C_macro_f1_1to4"] >= 0.40).all()))

### Extension vs required: Part 5 comparison

In [ ]:
KEY = ["model", "strategy", "budget"]
SHOW = ["C_label1_f1", "C_macro_f1_1to4", "C_strict_macro_f1_2to4", "A_test_macro_f1", "B_test_macro_f1"]
p5_compare = results_df[KEY + SHOW].merge(ext_results_df[KEY + SHOW], on=KEY, suffixes=("_unigram", "_bigram"))
for m in SHOW:
    p5_compare[f"{m}_change"] = p5_compare[f"{m}_bigram"] - p5_compare[f"{m}_unigram"]
p5_compare.to_csv("results/extension/part5_comparison.csv", index=False)
display(p5_compare[KEY + [f"{m}_{s}" for m in ["C_label1_f1", "C_macro_f1_1to4"] for s in ["unigram", "bigram", "change"]]
                   + [f"{m}_change" for m in ["A_test_macro_f1", "B_test_macro_f1"]]].round(4))

In [ ]:
# Required (solid) vs extension (dashed), per strategy
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = dict(zip(STRATEGIES, ["tab:blue", "tab:orange", "tab:green"]))
for i, tag in enumerate(["LR", "MLP"]):
    for j, (metric, label) in enumerate([("C_label1_f1", "Data C label-1 (World) F1"),
                                         ("C_macro_f1_1to4", "Data C macro-F1 (labels 1-4)")]):
        ax = axes[i, j]
        for df, style, rep in [(results_df, "-", "unigram"), (ext_results_df, "--", "uni+bigram")]:
            d = df[df.model == tag]
            base = d[d.budget == 0].iloc[0]
            for s in STRATEGIES:
                ds = d[d.strategy == s].sort_values("budget")
                ax.plot([0] + ds.budget.tolist(), [base[metric]] + ds[metric].tolist(),
                        linestyle=style, marker="o", color=colors[s], label=f"{s} ({rep})")
        ax.set_xscale("symlog", linthresh=50)
        ax.set_xticks(TICKS)
        ax.set_xticklabels(TICKS)
        ax.set_xlabel("Added Data C training examples")
        ax.set_ylabel(label)
        ax.set_title(f"{tag}: {label}")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig("results/extension/part5_comparison_curves.png", dpi=150)
plt.show()

In [ ]:
# Where true Data C World (label 1) articles go, required vs extension Part 3 models
rows = []
for tag in ["lr", "mlp"]:
    for rep, cms in [("unigram", conf_mats), ("uni+bigram", ext_conf_mats)]:
        world_row = np.array(cms[f"{tag}_baseline_0"])[0]  # true label 1
        rows.append({"model": tag.upper(), "representation": rep,
                     **{f"pred_{LABELS.loc[l, 'Set A']}": int(world_row[l - 1]) for l in CLASS_IDS}})
display(pd.DataFrame(rows))